# 🛣️ Guardian Road
### Building an AI Safety Shield Around a Fallen Rider

A motorcycle falls in the centre lane. The road is wet. Traffic is arriving at 72 km/h, and the
closest vehicle is 118 metres away.

**This notebook asks how the road itself can prevent the next collision.** It decides where warnings
begin, how speed falls, where vehicles merge, and how to preserve safe routes for helpers and an
ambulance.

> ⚠️ Everything here is simulated. It is not a traffic controller, medical device or emergency
> system. It detects a road obstruction—not an injury—and may not control a real road.

### Research question

Can a physics-guided AI controller reduce simulated secondary-collision risk while producing less
severe braking and less traffic delay than a fixed-distance warning system?

### What we will build

1. A three-lane synthetic road and imperfect traffic.
2. A temporal fallen-rider detector with honest lookalikes.
3. A physics floor for safe warning distance.
4. Cost-sensitive regressors that may add—but never remove—safety margin.
5. Dynamic studs, speed limits, merge control, safe paths and ambulance access.
6. A seven-controller benchmark, sensor-failure test and staged reopening procedure.

### Contents

1. Build the road
2. Simulate normal traffic
3. Introduce the crash
4. Detect the fallen rider over time
5. Calculate stopping distance
6. Test a fixed warning
7. Predict dynamic warning distance
8. Make late warnings cost more
9. Control connected road studs
10. Model imperfect drivers
11. Choose the merge direction
12. Preserve ambulance access
13. Find a safe bystander path
14. Test multiple hazards and failures
15. Restore normal traffic

## Setup

The notebook is standalone. It contains the simulation source inline and uses fixed seeds so the figures and prose agree each time.

In [ ]:
# !pip install numpy pandas plotly scikit-learn nbformat
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier

np.random.seed(7)
print("Ready.")

In [ ]:
# Guardian Road simulation core — kept in guardian.py in the project
"""Deterministic teaching simulation for Guardian Road.

Everything here is synthetic. It is not a traffic controller or safety system.
"""
from __future__ import annotations

import heapq
import math
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import plotly.graph_objects as go

G = 9.81
LANES = ("Left", "Centre", "Right")
COLORS = dict(bg="#0e1117", panel="#161b22", cyan="#4fc3f7", amber="#ffb74d",
              red="#ef5350", green="#66bb6a", blue="#42a5f5", violet="#ba68c8",
              grey="#8b949e", white="#e6edf3")
VEHICLES = {
    "motorcycle": dict(length=2.2, decel=7.2, reaction=1.05, mass_factor=.85),
    "car": dict(length=4.5, decel=6.5, reaction=1.30, mass_factor=1.00),
    "bus": dict(length=12.0, decel=4.3, reaction=1.55, mass_factor=1.28),
    "truck": dict(length=15.0, decel=4.0, reaction=1.65, mass_factor=1.38),
}
WEATHER = {
    "Dry": dict(mu=.78, visibility=900, margin=1.00),
    "Wet": dict(mu=.48, visibility=500, margin=1.14),
    "Fog": dict(mu=.58, visibility=170, margin=1.24),
    "Night": dict(mu=.68, visibility=320, margin=1.12),
    "Heavy rain": dict(mu=.38, visibility=130, margin=1.34),
}


@dataclass
class Scene:
    speed_kmh: float = 72
    weather: str = "Wet"
    reaction_s: float = 1.4
    vehicle_type: str = "car"
    density: float = 34
    left_occupancy: float = .32
    right_occupancy: float = .78
    closest_m: float = 118
    gradient_pct: float = 0
    ambulance_eta_min: float = 9
    detection_confidence: float = .97
    radar_ok: bool = True
    camera_ok: bool = True
    studs_ok: float = 1.0
    communications_ok: bool = True


def stopping_distance(speed_kmh, reaction_s, mu, gradient_pct=0, vehicle_type="car"):
    """Reaction + friction-limited braking distance on a gentle grade."""
    v = speed_kmh / 3.6
    spec = VEHICLES[vehicle_type]
    grade = gradient_pct / 100
    effective_mu = max(.12, mu + grade)
    reaction = v * reaction_s
    friction_decel = effective_mu * G
    decel = min(friction_decel, spec["decel"])
    braking = v * v / (2 * decel)
    return reaction, braking, reaction + braking


def required_warning(scene: Scene, quantile_margin=1.0):
    w = WEATHER[scene.weather]
    reaction, braking, stop = stopping_distance(scene.speed_kmh, scene.reaction_s,
                                                w["mu"], scene.gradient_pct,
                                                scene.vehicle_type)
    density_margin = 1 + .0035 * max(scene.density - 18, 0)
    heavy_margin = VEHICLES[scene.vehicle_type]["mass_factor"]
    visibility_margin = 1 + max(0, 350 - w["visibility"]) / 900
    uncertainty = 18 + 18 * (1 - scene.detection_confidence)
    total = (stop * w["margin"] * density_margin * heavy_margin * visibility_margin
             * quantile_margin + uncertainty)
    return dict(reaction_m=reaction, braking_m=braking, base_stop_m=stop,
                required_m=float(np.clip(total, 60, 650)))


def merge_choice(scene: Scene):
    left = 1 - scene.left_occupancy
    right = 1 - scene.right_occupancy
    if scene.ambulance_eta_min <= 5:
        left -= .32  # preserve the left lane for the ambulance
    if max(scene.left_occupancy, scene.right_occupancy) > .92:
        return "Stop all traffic", "Neither adjacent lane has a safe receiving gap."
    if abs(left - right) < .08:
        return "Split traffic", "The two adjacent lanes have similar spare capacity."
    if left > right:
        return "Merge left", "The left lane has more usable capacity."
    return "Merge right", "The right lane has more usable capacity."


def controller(scene: Scene, mode="Safe AI + ambulance"):
    physics = required_warning(scene)
    fallback = (not scene.camera_ok or not scene.radar_ok or
                not scene.communications_ok or scene.detection_confidence < .72 or
                scene.studs_ok < .65)
    if mode == "No warning":
        warning = 0
    elif mode == "Static sign":
        warning = 80
    elif mode == "Fixed 150 m":
        warning = 150
    elif mode == "Physics minimum":
        warning = physics["required_m"]
    else:
        # Surrogate for a conservative, physics-guided learned residual.
        nonlinear = 14 * (scene.density / 60) ** 2 + 12 * scene.right_occupancy
        warning = max(physics["required_m"], physics["required_m"] + nonlinear - 6)
    direction, why = merge_choice(scene)
    if fallback:
        warning = max(warning, 420)
        direction, why = "Stop all traffic", "Sensor confidence is poor: conservative fallback."
    reserve = mode == "Safe AI + ambulance" and scene.ambulance_eta_min <= 10
    speed_limit = 30 if fallback else (40 if scene.weather in ("Wet", "Fog", "Heavy rain") else 50)
    if scene.speed_kmh <= 50 and not fallback:
        speed_limit = 30 if scene.weather == "Heavy rain" else 40
    return dict(mode=mode, warning_m=int(math.ceil(warning / 10) * 10),
                speed_limit=speed_limit, closed_lane="Centre", merge=direction,
                merge_reason=why, exclusion_m=45 if fallback else 30,
                reserve_ambulance=reserve, fallback=fallback,
                upstream_red=fallback or scene.closest_m < physics["required_m"] * .7,
                physics_min_m=round(physics["required_m"], 1))


def outcome(scene: Scene, action):
    req = required_warning(scene)["required_m"]
    deficit = max(req - action["warning_m"], 0)
    surplus = max(action["warning_m"] - req, 0)
    risk = 1 / (1 + np.exp(-(deficit / 22 - 2.2)))
    if action["upstream_red"]:
        risk *= .35
    decel = min(9.0, 2.0 + 7.0 * deficit / max(req, 1))
    disruption = surplus * .10 + scene.density * (1.2 if action["upstream_red"] else .32)
    if action["merge"] == "Stop all traffic":
        disruption += 48
    blockage = .08 + .38 * scene.left_occupancy
    if action["reserve_ambulance"]:
        blockage *= .22
    return dict(collision_probability=float(np.clip(risk, .004, .98)),
                max_deceleration=float(decel), traffic_delay_s=float(disruption),
                ambulance_blockage=float(np.clip(blockage, 0, 1)),
                minimum_clearance_m=float(max(scene.closest_m - req, -20)))


def build_warning_dataset(n=3500, seed=7):
    rng = np.random.default_rng(seed)
    kinds = np.array(list(VEHICLES))
    weather = np.array(list(WEATHER))
    rows = []
    for _ in range(n):
        s = Scene(speed_kmh=rng.uniform(30, 105), weather=str(rng.choice(weather)),
                  reaction_s=np.clip(rng.normal(1.45, .35), .7, 2.8),
                  vehicle_type=str(rng.choice(kinds, p=[.10, .67, .08, .15])),
                  density=rng.uniform(5, 70), left_occupancy=rng.uniform(.05, .98),
                  right_occupancy=rng.uniform(.05, .98), closest_m=rng.uniform(70, 650),
                  gradient_pct=rng.uniform(-4, 5), detection_confidence=rng.uniform(.72, 1))
        d = asdict(s)
        d.update(required_warning(s, quantile_margin=rng.uniform(.96, 1.12)))
        rows.append(d)
    return pd.DataFrame(rows)


def asymmetric_loss(y_true, y_pred, alpha=8, beta=1):
    err = np.asarray(y_true) - np.asarray(y_pred)
    return np.mean(np.where(err > 0, alpha * err, beta * -err))


def compare_controllers(scene: Scene):
    modes = ["No warning", "Static sign", "Fixed 150 m", "Physics minimum",
             "ML predictor", "Safe AI", "Safe AI + ambulance"]
    rows = []
    for mode in modes:
        a = controller(scene, mode)
        o = outcome(scene, a)
        rows.append(dict(Controller=mode, **a, **o))
    return pd.DataFrame(rows)


def detection_signals(kind="fallen_rider", seconds=16, fps=5, seed=7):
    rng = np.random.default_rng(seed)
    t = np.arange(0, seconds, 1 / fps)
    impact = 3.0
    after = t >= impact
    person_road = np.zeros_like(t)
    horizontal = np.zeros_like(t)
    still = np.zeros_like(t)
    separation = np.zeros_like(t)
    trajectory_change = np.zeros_like(t)
    if kind == "fallen_rider":
        person_road[after], horizontal[after], separation[after] = .94, .9, .96
        still[t >= 4], trajectory_change[t >= 3.3] = .94, .72
    elif kind == "debris":
        horizontal[after], still[after], trajectory_change[t >= 4] = .6, .98, .55
    elif kind == "crossing":
        person_road[(t >= 3) & (t < 7)] = .92
        horizontal[(t >= 3) & (t < 7)] = .12
    elif kind == "worker":
        person_road[after], horizontal[after], still[after] = .82, .56, .60
    elif kind == "shadow":
        horizontal[(t >= 3) & (t < 4.2)] = .7
    noise = lambda: rng.normal(0, .045, len(t))
    df = pd.DataFrame(dict(t=t, person_road=np.clip(person_road + noise(), 0, 1),
                           horizontal=np.clip(horizontal + noise(), 0, 1),
                           still=np.clip(still + noise(), 0, 1),
                           separation=np.clip(separation + noise(), 0, 1),
                           trajectory_change=np.clip(trajectory_change + noise(), 0, 1)))
    df["kind"] = kind
    return df


def detection_probability(df):
    z = (-5 + 1.4 * df.person_road + 1.25 * df.horizontal + 1.6 * df.still
         + 2.0 * df.separation + 1.15 * df.trajectory_change)
    raw = 1 / (1 + np.exp(-z))
    return raw.rolling(5, min_periods=1).mean()


def road_zones(action):
    w = action["warning_m"]
    return pd.DataFrame([
        ("Awareness", w, max(w - 150, 0), "AMBER FLASHING"),
        ("Reduce speed", max(w - 150, 0), 250, f"LIMIT {action['speed_limit']}"),
        ("Merge", min(250, w), 100, action["merge"].upper()),
        ("Exclusion", 100, action["exclusion_m"], "CENTRE LANE CLOSED"),
        ("Rider protection", action["exclusion_m"], 0, "RED SHIELD"),
    ], columns=["zone", "upstream_m", "downstream_m", "pattern"])


def hazard_grid(active_lanes=(0, 1, 2), rider=(8, 22), reserve_lane=0):
    rows, cols = 18, 36
    cost = np.ones((rows, cols))
    for lane in active_lanes:
        y0 = 3 + lane * 4
        cost[y0:y0 + 4, :] = 60
    cost[rider[0]-2:rider[0]+3, rider[1]-3:rider[1]+4] = 400
    cost[3 + reserve_lane * 4:7 + reserve_lane * 4, :] = 25
    cost[:3, :] = 2
    cost[15:, :] = 2
    return cost


def plan_path(cost, start=(17, 2), goal=(8, 22)):
    pq = [(0, start)]
    prev, dist = {}, {start: 0}
    while pq:
        d, node = heapq.heappop(pq)
        if node == goal:
            break
        if d != dist[node]:
            continue
        for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
            nxt = node[0] + dr, node[1] + dc
            if not (0 <= nxt[0] < cost.shape[0] and 0 <= nxt[1] < cost.shape[1]):
                continue
            nd = d + float(cost[nxt])
            if nd < dist.get(nxt, math.inf):
                dist[nxt], prev[nxt] = nd, node
                heapq.heappush(pq, (nd, nxt))
    if goal not in dist:
        return []
    path, node = [goal], goal
    while node != start:
        node = prev[node]
        path.append(node)
    return path[::-1]


def _layout(fig, height=420, **kwargs):
    fig.update_layout(height=height, paper_bgcolor=COLORS["bg"], plot_bgcolor=COLORS["bg"],
                      font_color=COLORS["white"], margin=dict(l=45,r=20,t=55,b=40),
                      legend=dict(bgcolor="rgba(0,0,0,0)"), **kwargs)
    fig.update_xaxes(gridcolor="#21262d")
    fig.update_yaxes(gridcolor="#21262d")
    return fig


def fig_road(scene: Scene, action, closest=None):
    closest = scene.closest_m if closest is None else closest
    w = max(action["warning_m"], 200)
    fig = go.Figure()
    for lane in range(3):
        y0 = lane
        fig.add_shape(type="rect", x0=0, x1=w, y0=y0, y1=y0+1,
                      fillcolor="#161b22", line=dict(color="#30363d"))
    for x in np.arange(20, w, 25):
        colour = COLORS["amber"] if x > 100 else COLORS["red"]
        fig.add_scatter(x=[x], y=[1.5], mode="markers", marker=dict(color=colour, size=8),
                        showlegend=False, hovertext="connected road stud")
    fig.add_scatter(x=[0], y=[1.5], mode="markers+text", text=["fallen rider"],
                    textposition="top center", marker=dict(color=COLORS["red"], size=18),
                    name="rider")
    xs = [min(closest, w*.94), min(closest+55, w*.98), min(closest+105, w*.99)]
    ys = [1.5, .5, 2.5]
    fig.add_scatter(x=xs, y=ys, mode="markers+text", text=[f"{scene.speed_kmh:.0f} km/h","car","truck"],
                    textposition="top center", marker=dict(color=COLORS["cyan"], size=[15,12,18]),
                    name="approaching traffic")
    fig.add_vrect(x0=0, x1=action["exclusion_m"], fillcolor=COLORS["red"], opacity=.18,
                  annotation_text="exclusion", annotation_font_color="white")
    fig.add_vline(x=action["warning_m"], line_color=COLORS["amber"], line_dash="dash",
                  annotation_text="warning begins", annotation_font_color="white")
    fig.update_yaxes(tickvals=[.5,1.5,2.5], ticktext=list(LANES), range=[0,3])
    fig.update_xaxes(autorange="reversed", title="metres upstream from rider")
    return _layout(fig, 360, title=f"{action['merge']} · centre lane closed · limit {action['speed_limit']} km/h")


def fig_tradeoff(scene: Scene):
    rows=[]
    req=required_warning(scene)["required_m"]
    for d in np.arange(50, 501, 10):
        a=controller(scene, "Safe AI")
        a["warning_m"]=d
        o=outcome(scene,a)
        rows.append((d,o["collision_probability"],o["max_deceleration"],o["traffic_delay_s"]))
    df=pd.DataFrame(rows,columns=["warning","risk","braking","delay"])
    fig=go.Figure()
    fig.add_scatter(x=df.warning,y=df.risk,name="collision probability",line=dict(color=COLORS["red"],width=3))
    fig.add_scatter(x=df.warning,y=df.delay/100,name="traffic delay (÷100)",line=dict(color=COLORS["amber"],width=3))
    fig.add_vline(x=req,line_dash="dash",line_color=COLORS["green"],annotation_text="physics minimum")
    return _layout(fig,title="Safety improves with distance; disruption grows",xaxis_title="warning distance (m)")


def fig_detection():
    fig=go.Figure()
    for kind,colour in zip(("fallen_rider","debris","crossing","worker","shadow"),
                           (COLORS["red"],COLORS["violet"],COLORS["green"],COLORS["amber"],COLORS["grey"])):
        d=detection_signals(kind)
        fig.add_scatter(x=d.t,y=detection_probability(d),name=kind.replace("_"," "),line=dict(color=colour,width=2.4))
    fig.add_hline(y=.72,line_dash="dash",line_color="white",annotation_text="confirmation threshold")
    return _layout(fig,title="A sequence separates the fallen rider from lookalikes",xaxis_title="seconds",yaxis_title="obstruction probability")


def fig_hazard(cost, path):
    fig=go.Figure(go.Heatmap(z=np.log10(cost),colorscale="Inferno",showscale=False))
    if path:
        fig.add_scatter(x=[p[1] for p in path],y=[p[0] for p in path],mode="lines",
                        line=dict(color=COLORS["green"],width=5),name="protected bystander route")
    return _layout(fig,360,title="Route shown only after traffic protection is established",
                   xaxis_title="distance along segment",yaxis_title="across the road")


## 1 · Build the road

The road is 650 metres long and has three lanes. The rider is the origin: every approaching vehicle
has a positive upstream distance. This makes the most important measurement readable—how much road
remains before the obstruction.

The drawing is not decoration. Its warning boundary, studs, lanes and vehicles are generated from
the same controller state used in the score table.

In [ ]:
scene = Scene()
action = controller(scene, "Safe AI + ambulance")
fig_road(scene, action).show()
pd.DataFrame([asdict(scene)]).T.rename(columns={0:"value"})

**Read it like this.** The red area is local. The amber warning begins far upstream. A safe design creates time gradually instead of demanding sudden braking beside the rider.

## 2 · Simulate normal traffic

Every driver has position, speed, reaction delay, a braking limit and a willingness to change lane.
Real traffic is heterogeneous: a truck cannot copy a motorcycle's braking, and a distracted driver
does not react like a connected vehicle.

In [ ]:
rng=np.random.default_rng(7)
profiles={
 "alert":(1.0,.94), "distracted":(2.1,.58), "elderly":(1.7,.82),
 "connected autonomous":(.35,.99), "heavy-truck":(1.65,.78),
 "aggressive":(1.2,.48), "following too closely":(1.45,.72)}
normal=[]
for i in range(90):
    driver=str(rng.choice(list(profiles)))
    reaction,compliance=profiles[driver]
    kind="truck" if driver=="heavy-truck" else str(rng.choice(["car","car","car","bus","motorcycle"]))
    normal.append(dict(vehicle_id=i,type=kind,lane=int(rng.integers(0,3)),
        position_m=float(rng.uniform(40,650)),speed_mps=float(rng.normal(20,3)),
        reaction_time_s=float(max(.3,rng.normal(reaction,.16))),
        max_safe_decel=VEHICLES[kind]["decel"],connected=driver=="connected autonomous",
        compliance_probability=compliance,driver=driver))
traffic=pd.DataFrame(normal).sort_values("position_m")
traffic.head(10)

In [ ]:
traffic["headway_s"] = traffic.groupby("lane")["position_m"].diff().abs() / traffic.speed_mps
traffic.groupby("lane").agg(vehicles=("vehicle_id","count"),mean_speed=("speed_mps","mean"),
                             median_headway=("headway_s","median"))

**The important imperfection.** Compliance is a probability, not a switch. Later experiments must survive drivers who respond late or refuse the first merge opportunity.

## 3 · Introduce the crash

At three seconds the rider and motorcycle separate. The rider remains in the centre lane while
traffic continues upstream. Camera signals are noisy, and useful evidence appears at different
times—not on one magical impact frame.

In [ ]:
incident=detection_signals("fallen_rider")
incident.head(25).tail(8)

In [ ]:
fig=go.Figure()
for name,colour in zip(["person_road","horizontal","still","separation","trajectory_change"],
                       [COLORS["cyan"],COLORS["amber"],COLORS["green"],COLORS["red"],COLORS["violet"]]):
    fig.add_scatter(x=incident.t,y=incident[name],name=name.replace("_"," "),line=dict(color=colour))
_layout(fig,title="What the camera can honestly report",xaxis_title="seconds",yaxis_title="noisy signal").show()

## 4 · Detect the fallen rider over time

A fallen rider, debris, a worker, a crossing pedestrian and a shadow can share one frame. The
sequence matters: separation from a motorcycle, persistent road occupancy, stillness, horizontal
orientation and nearby trajectory change.

The output is an **obstruction probability**. The medical condition remains unknown.

In [ ]:
fig_detection().show()

In [ ]:
det=[]
for kind in ["fallen_rider","debris","crossing","worker","shadow"]:
    d=detection_signals(kind)
    one_frame=((d.person_road>.7)&(d.horizontal>.5)).astype(float)
    temporal=((d.person_road.rolling(12,min_periods=1).mean()>.55)&
              (d.still.rolling(12,min_periods=1).mean()>.45)).astype(float)
    sequence=detection_probability(d)
    det.append(dict(scene=kind,one_frame=float(one_frame.max()),fixed_rule=float(temporal.max()),
                    sequence_peak=float(sequence.max()),confirmed=bool((sequence>.72).rolling(5).sum().max()>=5)))
pd.DataFrame(det).set_index("scene")

**Why call before certainty?** A conservative traffic warning is reversible. Waiting for a medical conclusion is both unnecessary and outside the system's authority.

## 5 · Calculate stopping distance

The engineering baseline is a sum—not a subtraction:

\[
D_{stop}=D_{reaction}+D_{braking},\quad D_{reaction}=vt_r,\quad
D_{braking}=\frac{v^2}{2\mu g}
\]

Wet friction, visibility, vehicle type, density, slope and uncertainty add margin. This physical
minimum remains below every learned controller.

In [ ]:
d=required_warning(scene)
pd.Series(d,name="metres").round(1)

In [ ]:
rows=[]
for speed in [30,50,70,90]:
    for weather in WEATHER:
        s=Scene(speed_kmh=speed,weather=weather)
        rows.append(dict(speed_kmh=speed,weather=weather,required_m=required_warning(s)["required_m"]))
stop_table=pd.DataFrame(rows)
stop_table.pivot(index="speed_kmh",columns="weather",values="required_m").round(0)

## 6 · Test a fixed warning

Now give every incident the same answer: warn at 150 m, reduce to 40 km/h and close the centre lane.
It is simple and explainable. It is also blind to the road in front of it.

In [ ]:
tests=[Scene(speed_kmh=90,weather="Wet",vehicle_type="truck",reaction_s=2.0),
       Scene(speed_kmh=72,weather="Heavy rain",reaction_s=1.8),
       Scene(speed_kmh=50,weather="Dry",reaction_s=1.0,density=8),
       Scene(speed_kmh=70,weather="Fog",reaction_s=1.5,density=68)]
fixed=[]
for s in tests:
    a=controller(s,"Fixed 150 m"); o=outcome(s,a)
    fixed.append(dict(speed=s.speed_kmh,weather=s.weather,vehicle=s.vehicle_type,
                      physics_min=a["physics_min_m"],warning=a["warning_m"],
                      unsafe=a["warning_m"]<a["physics_min_m"],risk=o["collision_probability"],delay=o["traffic_delay_s"]))
pd.DataFrame(fixed)

**Two opposite failures.** A fixed boundary can be dangerously late for a wet-road truck and wastefully early for light, slow, dry traffic.

## 7 · Predict dynamic warning distance

We generate thousands of synthetic situations from the same physical world. Four regressors predict
the required distance. The split is made before fitting, and the held-out test pile is touched only
for the final table.

In [ ]:
data=build_warning_dataset(3500)
features=["speed_kmh","weather","reaction_s","vehicle_type","density","left_occupancy",
          "right_occupancy","closest_m","gradient_pct","detection_confidence"]
cat=["weather","vehicle_type"]; num=[c for c in features if c not in cat]
X_train,X_test,y_train,y_test=train_test_split(data[features],data.required_m,test_size=.25,random_state=7)
prep=ColumnTransformer([("num",StandardScaler(),num),("cat",OneHotEncoder(handle_unknown="ignore"),cat)])
models={"linear":LinearRegression(),"tree":DecisionTreeRegressor(max_depth=8,min_samples_leaf=18,random_state=7),
        "forest":RandomForestRegressor(n_estimators=140,max_depth=14,min_samples_leaf=4,n_jobs=-1,random_state=7),
        "neural network":MLPRegressor(hidden_layer_sizes=(48,24),max_iter=350,early_stopping=True,random_state=7)}
predictions={}; fitted={}
for name,model in models.items():
    pipe=make_pipeline(prep,model); pipe.fit(X_train,y_train)
    fitted[name]=pipe; predictions[name]=pipe.predict(X_test)
pd.DataFrame([{"model":n,"MAE metres":mean_absolute_error(y_test,p),
               "unsafe predictions":int((p<y_test).sum()),
               "worst underprediction":float(np.max(y_test-p))} for n,p in predictions.items()]).set_index("model").round(2)

## 8 · Make late warnings cost more

MAE says 20 m early and 20 m late are equal. The road does not. We use

\[
L=\begin{cases}\alpha|e|,&\text{warning too late}\\\beta|e|,&\text{warning too early}\end{cases},
\quad \alpha=8,\ \beta=1.
\]

A deployment controller then applies a physical guard: `max(model prediction, physics minimum)`.

In [ ]:
rows=[]
physics_floor=np.array([required_warning(Scene(speed_kmh=r.speed_kmh,weather=r.weather,
 reaction_s=r.reaction_s,vehicle_type=r.vehicle_type,density=r.density,left_occupancy=r.left_occupancy,
 right_occupancy=r.right_occupancy,closest_m=r.closest_m,gradient_pct=r.gradient_pct,
 detection_confidence=r.detection_confidence))["required_m"] for r in X_test.itertuples()])
for name,p in predictions.items():
    guarded=np.maximum(p,physics_floor)
    rows.append(dict(model=name,MAE=mean_absolute_error(y_test,p),asymmetric=asymmetric_loss(y_test,p),
                     unsafe_before=int((p<physics_floor).sum()),unsafe_after=int((guarded<physics_floor).sum())))
pd.DataFrame(rows).set_index("model").round(2)

In [ ]:
# Cost-sensitive calibration: use a training residual quantile, not the test answers.
# alpha/(alpha+beta)=8/9 asks for a deliberately conservative prediction quantile.
alpha,beta=8,1
base=fitted["forest"]
train_pred=base.predict(X_train)
safety_offset=float(np.quantile(y_train-train_pred,alpha/(alpha+beta)))
raw=predictions["forest"]
cost_sensitive=raw+safety_offset
guarded=np.maximum(cost_sensitive,physics_floor)
pd.Series({"learned safety offset (m)":safety_offset,
           "raw asymmetric loss":asymmetric_loss(y_test,raw,alpha,beta),
           "cost-sensitive asymmetric loss":asymmetric_loss(y_test,cost_sensitive,alpha,beta),
           "unsafe after physical guard":int((guarded<physics_floor).sum())}).round(2)

**The guard is the architecture.** Better training can reduce error. It cannot replace the minimum stopping-distance constraint.

## 9 · Control connected road studs

One distance becomes five visible zones: awareness, speed reduction, merge, exclusion and rider
protection. The road changes shape when the sidebar conditions—or the code below—change.

In [ ]:
road_zones(action)

In [ ]:
fig_road(scene,action).show()

## 10 · Model imperfect drivers

Perfect compliance makes every controller look safe. We replay driver profiles with different
reaction distributions, braking limits, headways and compliance probabilities.

In [ ]:
rng=np.random.default_rng(11); driver_rows=[]
for driver,(mean_rt,compliance) in profiles.items():
    risks=[]
    for _ in range(250):
        typ="truck" if driver=="heavy-truck" else "car"
        s=Scene(reaction_s=max(.3,rng.normal(mean_rt,.18)),vehicle_type=typ,
                density=rng.uniform(15,60),weather=str(rng.choice(list(WEATHER))))
        a=controller(s,"Safe AI")
        if rng.random()>compliance: a["warning_m"]*=.68
        risks.append(outcome(s,a)["collision_probability"])
    driver_rows.append(dict(driver=driver,mean_risk=np.mean(risks),p95_risk=np.quantile(risks,.95)))
pd.DataFrame(driver_rows).set_index("driver").round(3)

## 11 · Choose the merge direction

Closing the centre lane does not automatically mean “merge left.” The receiving lane needs capacity,
a usable gap and no conflict with the responder plan. If neither side is safe, stopping traffic is a
valid answer.

In [ ]:
merge_tests=[]
for left,right,eta in [(.25,.82,9),(.84,.22,9),(.42,.47,9),(.95,.96,9),(.25,.82,3)]:
    s=Scene(left_occupancy=left,right_occupancy=right,ambulance_eta_min=eta)
    choice,reason=merge_choice(s)
    merge_tests.append(dict(left_occupancy=left,right_occupancy=right,ambulance_eta=eta,choice=choice,reason=reason))
pd.DataFrame(merge_tests)

## 12 · Preserve ambulance access

When the ambulance is dispatched, the objective adds access time and route-blockage risk. The safe
controller reserves a lane early instead of trying to empty it when blue lights are already at the queue.

In [ ]:
amb=[]
for density in [15,35,55,70]:
    s=Scene(density=density,left_occupancy=min(.92,density/80),ambulance_eta_min=4)
    for mode in ["Safe AI","Safe AI + ambulance"]:
        a=controller(s,mode); o=outcome(s,a)
        amb.append(dict(density=density,plan=mode,reserved=a["reserve_ambulance"],
                        blockage_probability=o["ambulance_blockage"],traffic_delay=o["traffic_delay_s"]))
pd.DataFrame(amb).pivot(index="density",columns="plan",values="blockage_probability").round(3)

## 13 · Find a safe bystander path

The shortest geometric line crosses moving traffic. Dijkstra instead minimizes accumulated risk.
The path may be displayed only after the controller confirms the relevant traffic protection.

In [ ]:
cost=hazard_grid(active_lanes=() if action["upstream_red"] else (0,2),reserve_lane=0)
path=plan_path(cost)
fig_hazard(cost,path).show()
print("Route cells:",len(path),"  accumulated risk cost:",round(sum(cost[p] for p in path),1))

**Hard rule.** No green route is shown through an active lane. If traffic protection is unverified, the instruction is to wait on the shoulder.

## 14 · Test hazards and sensor failures

Heavy rain, occlusion, failed radar, missing communications and dead studs are not rare exceptions;
they are part of the operating world. Poor confidence bypasses optimization and expands the zone.

In [ ]:
failures=[("healthy",Scene()),("camera blocked",Scene(camera_ok=False)),
          ("radar lost",Scene(radar_ok=False)),("stud failures",Scene(studs_ok=.45)),
          ("control link lost",Scene(communications_ok=False)),
          ("heavy rain + low confidence",Scene(weather="Heavy rain",detection_confidence=.58))]
rows=[]
for label,s in failures:
    a=controller(s); o=outcome(s,a)
    rows.append(dict(test=label,fallback=a["fallback"],warning_m=a["warning_m"],
                     signal_red=a["upstream_red"],merge=a["merge"],risk=o["collision_probability"]))
pd.DataFrame(rows).set_index("test")

**Read the direction of failure.** Missing confidence makes the protected zone larger and the traffic decision more conservative.

## 15 · Restore normal traffic

Reopening is a guarded state machine:

1. An authorized responder confirms lane clearance.
2. The red exclusion zone is removed.
3. A temporary reduced-speed zone remains.
4. Queued traffic is released gradually.
5. Signals return to normal.
6. Failed studs and sensors are recorded for maintenance.

The optimizer cannot declare the lane clear by itself.

In [ ]:
def restoration_state(responder_clear, occupancy, hardware_ok):
    if not responder_clear: return "PROTECTED — wait for authorized clearance"
    if occupancy>.55: return "METERED RELEASE — keep reduced speed"
    if not hardware_ok: return "LIMITED REOPENING — record and isolate failed equipment"
    return "NORMAL — signals restored after gradual release"

pd.DataFrame([{"responder_clear":c,"queue_occupancy":q,"hardware_ok":h,
               "state":restoration_state(c,q,h)}
              for c,q,h in [(False,.8,True),(True,.8,True),(True,.3,False),(True,.3,True)]])

## Final benchmark

Seven controllers face the opening scenario. Secondary-collision probability remains separate from
braking, delay and ambulance blockage so a convenient average cannot hide an unsafe decision.

In [ ]:
scoreboard=compare_controllers(scene)
scoreboard[["Controller","warning_m","collision_probability","max_deceleration",
            "traffic_delay_s","ambulance_blockage","fallback"]].set_index("Controller").round(3)

In [ ]:
fig_tradeoff(scene).show()

## What this simulation may claim

It can compare controllers inside its invented world. It can show why warning distance must change,
why underprediction deserves a larger loss, why driver diversity matters and why physical constraints
must sit after AI optimization.

It cannot prove that a real deployment prevents collisions. That requires validated sensors, calibrated
vehicle dynamics, human-factors studies, road-authority approval, cybersecurity engineering, field trials
and emergency-service governance.

### Rules that do not move

- Warn conservatively before optimizing.
- Never predict below the physics minimum.
- Never diagnose the rider.
- Never show a bystander path through active traffic.
- Give traffic control and emergency services authority over the optimizer.
- Enter a conservative fallback when observations or actuators are unreliable.
- Reopen only after authorized clearance.